# Trustless-by-Design NTN Framework — Results Visualization
Generates all paper figures from experiment JSON files.
Run all cells sequentially. Figures saved to `results/figures/`.

In [1]:
import json, os, sys
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sys.path.insert(0, '..')
os.makedirs('../results/figures', exist_ok=True)

# IEEE-style plot settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'lines.linewidth': 1.5,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

COLORS = {
    'proposed':    '#1f77b4',
    'centralized': '#d62728',
    'static_dlt':  '#ff7f0e',
    'zt_auth':     '#2ca02c',
    'uav_fl':      '#9467bd',
}
LABELS = {
    'proposed':    'Proposed (DLT+ZTA)',
    'centralized': 'Centralized Trust',
    'static_dlt':  'Static DLT (NxGenT)',
    'zt_auth':     'ZT Auth-Only',
    'uav_fl':      'UAV+FL Blockchain',
}
MODELS = ['proposed','centralized','static_dlt','zt_auth','uav_fl']

def load(name):
    with open(f'../results/data/{name}.json') as f:
        return json.load(f)

print('Environment ready. Loading data...')
d1 = load('exp1_trust_dynamics')
d2 = load('exp2_delay_partition')
d3 = load('exp3_crypto_overhead')
d4 = load('exp4_privacy_utility')
d5 = load('exp5_scalability')
d6 = load('exp6_attack_comparison')
print('All 6 experiment files loaded.')

Environment ready. Loading data...
All 6 experiment files loaded.


## Figure 1 — Trust Score Evolution (Exp1)

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))

# Average trust curves over 30 runs
num_rounds = 500
rounds = np.arange(1, num_rounds + 1)

for m in MODELS:
    # Malicious node curve
    curves = [r['trust_curve_malicious'][m] for r in d1['runs']
              if len(r['trust_curve_malicious'][m]) == num_rounds]
    if curves:
        arr = np.array(curves)
        mean = arr.mean(axis=0)
        std  = arr.std(axis=0)
        axes[0].plot(rounds, mean, color=COLORS[m], label=LABELS[m])
        axes[0].fill_between(rounds, mean-std, mean+std,
                              color=COLORS[m], alpha=0.12)

    # Legitimate node curve
    curves_l = [r['trust_curve_legit'][m] for r in d1['runs']
                if len(r['trust_curve_legit'][m]) == num_rounds]
    if curves_l:
        arr_l = np.array(curves_l)
        mean_l = arr_l.mean(axis=0)
        axes[1].plot(rounds, mean_l, color=COLORS[m], label=LABELS[m])

for ax in axes:
    ax.axhline(0.4, color='black', linestyle='--', linewidth=1, label='Threshold theta=0.4')
    ax.set_xlabel('Simulation Round')
    ax.set_ylabel('Trust Score T_0j')
    ax.set_ylim(-0.02, 1.05)
    ax.set_xlim(1, num_rounds)

# Mark betrayal point
axes[0].axvline(150, color='gray', linestyle=':', linewidth=1)
axes[0].text(155, 0.92, 'Betrayal\nstart', fontsize=8, color='gray')
axes[0].set_title('(a) Malicious Node Trust Decay')
axes[1].set_title('(b) Legitimate Node Trust Stability')

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.18), frameon=True)
plt.tight_layout()
plt.savefig('../results/figures/fig1_trust_evolution.pdf')
plt.savefig('../results/figures/fig1_trust_evolution.png')
plt.show()
print('Fig1 saved.')

Fig1 saved.


C:\Users\dell\AppData\Local\Temp\ipykernel_2680\2316774721.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 2 — Detection Latency (Exp1)

In [3]:
fig, ax = plt.subplots(figsize=(5.5, 3.2))

means = [d1['summary'][m]['mean_detection_rounds'] for m in MODELS]
ci95  = [d1['summary'][m]['ci95'] for m in MODELS]
colors = [COLORS[m] for m in MODELS]
xlabels = [LABELS[m] for m in MODELS]

bars = ax.barh(range(len(MODELS)), means, xerr=ci95,
               color=colors, capsize=4, alpha=0.85,
               error_kw={'elinewidth': 1.2})

for i, (bar, mean, ci) in enumerate(zip(bars, means, ci95)):
    ax.text(mean + ci + 2, i, f'{mean:.0f}', va='center', fontsize=9)

ax.set_yticks(range(len(MODELS)))
ax.set_yticklabels(xlabels)
ax.set_xlabel('Detection Latency (rounds, lower is better)')
ax.set_title('Malicious Node Detection Latency (30 runs, 95% CI)')
ax.set_xlim(0, 280)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../results/figures/fig2_detection_latency.pdf')
plt.savefig('../results/figures/fig2_detection_latency.png')
plt.show()
print('Fig2 saved.')

Fig2 saved.


C:\Users\dell\AppData\Local\Temp\ipykernel_2680\2976647243.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 3 — Partition Resilience (Exp2)

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))

# Panel (a): Trust divergence over partition rounds
all_divs = {}
for run in d2['runs']:
    for pt in run['divergence_curve']:
        r = pt['round']
        all_divs.setdefault(r, []).append(pt['divergence'])

if all_divs:
    rounds_div = sorted(all_divs.keys())
    means_div  = [np.mean(all_divs[r]) for r in rounds_div]
    std_div    = [np.std(all_divs[r]) for r in rounds_div]
    axes[0].plot(rounds_div, means_div, color=COLORS['proposed'], label='Proposed')
    axes[0].fill_between(rounds_div,
                          np.array(means_div)-np.array(std_div),
                          np.array(means_div)+np.array(std_div),
                          alpha=0.2, color=COLORS['proposed'])
axes[0].set_xlabel('Simulation Round')
axes[0].set_ylabel('Trust Divergence (MAD)')
axes[0].set_title('(a) Trust Divergence During Partition')

# Panel (b): F1 pre vs post partition
s2 = d2['summary']
categories = ['Pre-Partition', 'Post-Partition']
values = [s2['f1_pre_partition_mean'], s2['f1_post_partition_mean']]
bar_colors = [COLORS['proposed'], '#17becf']
bars = axes[1].bar(categories, values, color=bar_colors, alpha=0.85, width=0.4)
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.005,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylabel('F1 Score (malicious detection)')
axes[1].set_title('(b) Detection F1: Pre vs Post Partition')
axes[1].set_ylim(0, 0.5)
axes[1].text(0.5, 0.45, f'Improvement: +{abs(s2["f1_degradation"]):.3f}',
             ha='center', transform=axes[1].transAxes, fontsize=9,
             color='darkgreen')

plt.tight_layout()
plt.savefig('../results/figures/fig3_partition_resilience.pdf')
plt.savefig('../results/figures/fig3_partition_resilience.png')
plt.show()
print('Fig3 saved.')

Fig3 saved.


C:\Users\dell\AppData\Local\Temp\ipykernel_2680\1042863317.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 4 — Cryptographic Overhead (Exp3)

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))

p = d3['paillier']
n_contrib  = [r['n_contributors'] for r in p]
enc_t  = [r['enc_time_ms'] for r in p]
agg_t  = [r['agg_time_ms'] for r in p]
dec_t  = [r['dec_time_ms'] for r in p]
total_t= [r['total_time_ms'] for r in p]
size_kb= [r['ciphertext_size_kb'] for r in p]

# Stacked bar: enc + agg + dec
x = np.arange(len(n_contrib))
w = 0.55
axes[0].bar(x, enc_t,  w, label='Encryption', color='#1f77b4', alpha=0.85)
axes[0].bar(x, agg_t,  w, bottom=enc_t, label='Aggregation', color='#ff7f0e', alpha=0.85)
axes[0].bar(x, dec_t,  w, bottom=[e+a for e,a in zip(enc_t,agg_t)],
            label='Decryption', color='#2ca02c', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(n_contrib)
axes[0].set_xlabel('Number of Contributors')
axes[0].set_ylabel('Time (ms)')
axes[0].set_title('(a) Paillier HE Overhead (1024-bit)')
axes[0].legend(fontsize=8)

# Ciphertext size vs contributors
axes[1].plot(n_contrib, size_kb, 'o-', color='#9467bd', linewidth=1.8)
axes[1].set_xlabel('Number of Contributors')
axes[1].set_ylabel('Ciphertext Size (KB)')
axes[1].set_title('(b) Aggregate Ciphertext Size')
for nc, sk in zip(n_contrib, size_kb):
    axes[1].annotate(f'{sk:.1f}KB', (nc, sk),
                     textcoords='offset points', xytext=(0,6), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('../results/figures/fig4_crypto_overhead.pdf')
plt.savefig('../results/figures/fig4_crypto_overhead.png')
plt.show()
print('Fig4 saved.')

Fig4 saved.


C:\Users\dell\AppData\Local\Temp\ipykernel_2680\1412377609.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 5 — Privacy-Utility Tradeoff (Exp4)

In [6]:
fig, ax1 = plt.subplots(figsize=(5.5, 3.2))

s4 = d4['summary']
eps_vals = [s['epsilon']   for s in s4]
mae_vals = [s['mae_mean']  for s in s4]
mae_ci   = [s['mae_ci95']  for s in s4]
f1_vals  = [s['f1_mean']   for s in s4]
f1_ci    = [s['f1_ci95']   for s in s4]
baseline_f1 = d4['baseline_f1_no_dp']

ax2 = ax1.twinx()

ln1 = ax1.errorbar(eps_vals, mae_vals, yerr=mae_ci,
                    fmt='s-', color='#d62728', capsize=4, label='MAE (left axis)')
ln2 = ax2.errorbar(eps_vals, f1_vals,  yerr=f1_ci,
                    fmt='o-', color='#1f77b4', capsize=4, label='F1 Score (right axis)')
ax2.axhline(baseline_f1, color='#1f77b4', linestyle='--', linewidth=1,
            label=f'Baseline F1 (no DP) = {baseline_f1:.3f}')

ax1.set_xlabel('Privacy Budget epsilon (higher = less private)')
ax1.set_ylabel('Trust MAE', color='#d62728')
ax1.tick_params(axis='y', labelcolor='#d62728')
ax2.set_ylabel('Detection F1 Score', color='#1f77b4')
ax2.tick_params(axis='y', labelcolor='#1f77b4')
ax1.set_xscale('log')
ax1.set_title('Privacy-Utility Tradeoff (Gaussian DP Mechanism)')

lines = [ln1, ln2]
labels_ = [l.get_label() for l in lines]
ax1.legend(lines, labels_, loc='center right', fontsize=8)

plt.tight_layout()
plt.savefig('../results/figures/fig5_privacy_utility.pdf')
plt.savefig('../results/figures/fig5_privacy_utility.png')
plt.show()
print('Fig5 saved.')

Fig5 saved.


C:\Users\dell\AppData\Local\Temp\ipykernel_2680\3488203472.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 6 — F1 vs Malicious Fraction (Exp6)

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2))

FRACS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
s6 = d6['summary']

# Panel (a): F1 vs malicious fraction
for m in MODELS:
    f1s = [s6[str(f)][m]['f1_mean'] for f in FRACS]
    ci  = [s6[str(f)][m]['f1_ci95'] for f in FRACS]
    axes[0].plot([f*100 for f in FRACS], f1s,
                 'o-', color=COLORS[m], label=LABELS[m])
    axes[0].fill_between([f*100 for f in FRACS],
                          np.array(f1s)-np.array(ci),
                          np.array(f1s)+np.array(ci),
                          alpha=0.12, color=COLORS[m])

axes[0].set_xlabel('Malicious Node Fraction (%)')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('(a) Detection F1 vs Attack Density')
axes[0].legend(fontsize=7, loc='upper left')

# Panel (b): AUC-ROC vs malicious fraction
for m in MODELS:
    aucs = [s6[str(f)][m]['auc_mean'] for f in FRACS]
    axes[1].plot([f*100 for f in FRACS], aucs,
                 's--', color=COLORS[m], label=LABELS[m])

axes[1].axhline(0.5, color='gray', linestyle=':', linewidth=1, label='Random baseline')
axes[1].set_xlabel('Malicious Node Fraction (%)')
axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('(b) AUC-ROC vs Attack Density')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.savefig('../results/figures/fig6_attack_comparison.pdf')
plt.savefig('../results/figures/fig6_attack_comparison.png')
plt.show()
print('Fig6 saved.')

Fig6 saved.


C:\Users\dell\AppData\Local\Temp\ipykernel_2680\1441332516.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 7 — Scalability (Exp5)

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))

r5 = d5['results']
Ns    = [r['n_nodes']           for r in r5]
times = [r['mean_round_time_ms'] for r in r5]
mems  = [r['memory_delta_mb']   for r in r5]
ledger= [r['ledger_size_kb']/1024 for r in r5]

axes[0].plot(Ns, times, 'o-', color=COLORS['proposed'], linewidth=2)
# Fit reference line O(N)
slope = times[-1] / Ns[-1]
ref = [slope * n for n in Ns]
axes[0].plot(Ns, ref, 'k--', linewidth=1, alpha=0.6, label='O(N) reference')
for n, t in zip(Ns, times):
    axes[0].annotate(f'{t:.0f}ms', (n, t),
                     textcoords='offset points', xytext=(0, 6),
                     ha='center', fontsize=8)
axes[0].set_xlabel('Number of Nodes (N)')
axes[0].set_ylabel('Computation Time per Round (ms)')
axes[0].set_title('(a) Computation Scalability')
axes[0].legend(fontsize=8)

ax_mem = axes[1].twinx()
l1, = axes[1].plot(Ns, ledger, 's-', color='#ff7f0e', linewidth=2, label='Ledger Size (MB)')
l2, = ax_mem.plot(Ns, mems, '^--', color='#9467bd', linewidth=2, label='Memory Delta (MB)')
axes[1].set_xlabel('Number of Nodes (N)')
axes[1].set_ylabel('Ledger Size (MB)', color='#ff7f0e')
ax_mem.set_ylabel('Memory Usage (MB)', color='#9467bd')
axes[1].set_title('(b) Storage and Memory Scalability')
axes[1].legend([l1, l2], ['Ledger Size (MB)', 'Memory Delta (MB)'],
               fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('../results/figures/fig7_scalability.pdf')
plt.savefig('../results/figures/fig7_scalability.png')
plt.show()
print('Fig7 saved.')

Fig7 saved.


C:\Users\dell\AppData\Local\Temp\ipykernel_2680\2277658110.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary Table — Paper Table III

In [9]:
print('\n=== TABLE III: QUANTITATIVE COMPARISON @ 20% MALICIOUS NODES ===')
print(f'{"Model":<22} {"DetLat(rds)":>13} {"F1":>7} {"AUC":>7} {"FPR":>7}')
print('-' * 62)
for m in MODELS:
    det  = d1['summary'][m]
    f1s  = [r['f1_history'][m][-1] for r in d1['runs']]
    auc  = d6['summary']['0.2'][m]['auc_mean']
    fpr  = d6['summary']['0.2'][m]['fpr_mean']
    dl   = det['mean_detection_rounds']
    dlci = det['ci95']
    f1   = np.mean(f1s)
    print(f'{LABELS[m]:<22} {dl:>7.1f}+/-{dlci:<4.1f} {f1:>7.3f} {auc:>7.3f} {fpr:>7.3f}')

print('\n=== EXP3: PAILLIER HE OVERHEAD SUMMARY ===')
print(f'{"Contributors":<15} {"Total(ms)":>10} {"Size(KB)":>10}')
for r in d3['paillier']:
    print(f"{r['n_contributors']:<15} {r['total_time_ms']:>10.1f} {r['ciphertext_size_kb']:>10.2f}")

print('\n=== EXP4: DP PRIVACY-UTILITY TRADEOFF ===')
print(f'{"epsilon":<10} {"sigma":<10} {"MAE":<10} {"F1":<10}')
for s in d4['summary']:
    print(f"{s['epsilon']:<10} {s['sigma']:<10.3f} {s['mae_mean']:<10.4f} {s['f1_mean']:<10.3f}")

print('\nAll figures saved to results/figures/')
import os
figs = os.listdir('../results/figures')
for f in sorted(figs):
    print(f'  {f}')


=== TABLE III: QUANTITATIVE COMPARISON @ 20% MALICIOUS NODES ===
Model                    DetLat(rds)      F1     AUC     FPR
--------------------------------------------------------------
Proposed (DLT+ZTA)        35.9+/-5.6    0.236   0.520   0.268
Centralized Trust        219.8+/-16.3   0.091   0.623   0.000
Static DLT (NxGenT)      152.6+/-12.8   0.159   0.630   0.000
ZT Auth-Only              85.5+/-7.8    0.251   0.571   0.000
UAV+FL Blockchain         17.3+/-4.0    0.236   0.514   0.279

=== EXP3: PAILLIER HE OVERHEAD SUMMARY ===
Contributors     Total(ms)   Size(KB)
5                     81.1       1.25
10                   155.3       2.50
25                   385.2       6.25
50                   755.3      12.50
100                 1499.6      25.00
200                 3037.1      50.00

=== EXP4: DP PRIVACY-UTILITY TRADEOFF ===
epsilon    sigma      MAE        F1        
0.1        48.448     0.4994     0.291     
0.5        9.690      0.4905     0.284     
1.0        4.84